In [4]:
import os

from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv("../.env")

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

#### Zero Shot Prompting Method Orignal Output

In [5]:
# def get_completion(prompt: str, deployment_name: str = deployment_name) -> dict:
#     """Get a chat completion from Azure OpenAI.

#     Args:
#         prompt: User input prompt.
#         deployment_name: The deployment name you gave your model in Azure portal.

#     Returns:
#         Full response object as dict, or error dict on failure.
#     """
#     try:
#         messages = [{"role": "user", "content": prompt}]

#         response = client.chat.completions.create(
#             model=deployment_name,
#             messages=messages,
#             temperature=0.1,
#             top_p=0.8,
#             max_tokens=512, 
#         )

#         return response.model_dump()

#     except Exception as e:
#         return {"error": str(e)}

In [6]:
# prompts: list[str] = [
#     "Imagine you are a detective trying to solve a mystery.",
#     "You arrive at the crime scene and start looking for clues.",
#     "You find a strange object at the crime scene. What is it?",
#     "How does this object relate to the crime?",
#     "Who do you think is the suspect and why?",
# ]

In [7]:
# for prompt in prompts:
#     response = get_completion(prompt)
#     print(f"Prompt: {prompt}")
#     print(f"Response: {response}")
#     print()

In [8]:
#As seen above: Each call to get_completion(prompt) is stateless.
'''
So the model:
--Does NOT remember previous prompts
--Treats every line like a brand new conversation

That’s why:
It asks for context again
It forgets the “object”
It can’t identify the suspect

'''

'\nSo the model:\n--Does NOT remember previous prompts\n--Treats every line like a brand new conversation\n\nThat’s why:\nIt asks for context again\nIt forgets the “object”\nIt can’t identify the suspect\n\n'

#### Zero Shot Prompting Method

In [9]:
#First lets test and return only content
def get_completion(prompt, deployment_name=deployment_name):
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.choices[0].message.content  

    except Exception as e:
        return f"Error: {str(e)}"

In [7]:
for prompt in prompts:
    response = get_completion(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()

Prompt: Imagine you are a detective trying to solve a mystery.
Response: Absolutely! I’m ready to put on my detective hat. What’s the mystery we’re trying to solve?

Prompt: You arrive at the crime scene and start looking for clues.
Response: You arrive at the crime scene, a dimly lit room with signs of a struggle. The air is thick with tension. You begin your search for clues.

What do you want to examine first?

- The body
- The surrounding area
- Any objects nearby
- Ask witnesses or bystanders

Prompt: You find a strange object at the crime scene. What is it?
Response: Could you please provide more details or describe the object you found? That way, I can help identify it or suggest what it might be.

Prompt: How does this object relate to the crime?
Response: Could you please provide more details or specify which object you are referring to? That way, I can give you a more accurate explanation of how it relates to the crime.

Prompt: Who do you think is the suspect and why?
Respon

In [8]:
'''
1. First response → rich story
The model creates its own scenario:

Blackwood Manor
Missing manuscript
Defined suspects

Second prompt → completely new scene

“You arrive at the crime scene…”

Now suddenly:

New location: apartment
No manuscript
No suspects

> That means → memory is lost
Later prompts → confusion
Model says:
 “what object?”
 “missing context”

> Because:
It never saw previous outputs
We didn’t pass them back'''

'''
Summary of what happened:
first answer looks “smart” > Because our first prompt: “Imagine you are a detective…”

This is open-ended, so model: invents everything, drives the narrative

But later prompts:
depend on previous context
which is missing
'''

'\nSummary of what happened:\nfirst answer looks “smart” > Because our first prompt: “Imagine you are a detective…”\n\nThis is open-ended, so model: invents everything, drives the narrative\n\nBut later prompts:\ndepend on previous context\nwhich is missing\n'

#### Chain of thought prompting Method

In [9]:
#OPTION 3 — Maintain conversation history
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=512
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [10]:
messages = [
    {"role": "system", "content": "You are a detective solving a mystery step by step."}
]

prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

Prompt: Start the investigation.
Response: Understood. To begin the investigation, I need some initial information:

1. What is the nature of the mystery or crime?
2. Where and when did it occur?
3. Are there any known suspects, witnesses, or evidence so far?
4. Any other relevant details you can provide.

Please provide these details so I can start piecing together the case.

Prompt: You arrive at the crime scene.
Response: I arrive at the crime scene. 

Step 1: Assess the scene and secure it to avoid contamination of evidence.  
Step 2: Observe and document the environment—note the location, time, and any immediate visible clues.  
Step 3: Identify and speak to any witnesses or individuals present.  
Step 4: Begin collecting physical evidence—photographs, fingerprints, any objects out of place, etc.

Please describe the scene or provide details about what I see upon arrival.

Prompt: You find a strange object. Describe it.
Response: At the crime scene, I notice a strange object lying

In [11]:
'''
Our setup produced:

--Sequential reasoning across steps
--Evidence → inference → conclusion flow
--Consistent story memory
--Logical suspect identification

Output:
Pen found
Initials identified
Guest log matched
Suspect inferred

>> That is implicit chain-of-thought reasoning

What We did NOT strictly achieve > explicit CoT format, like:

“Step 1: …”
“Reasoning: …”
“Therefore: …”

Instead, the model:
--Embedded reasoning naturally inside narration
--Mixed story + logic
This is called:
>> Narrative reasoning, not strict CoT prompting
'''

'\nOur setup produced:\n\n--Sequential reasoning across steps\n--Evidence → inference → conclusion flow\n--Consistent story memory\n--Logical suspect identification\n\nOutput:\nPen found\nInitials identified\nGuest log matched\nSuspect inferred\n\n>> That is implicit chain-of-thought reasoning\n\nWhat We did NOT strictly achieve > explicit CoT format, like:\n\n“Step 1: …”\n“Reasoning: …”\n“Therefore: …”\n\nInstead, the model:\n--Embedded reasoning naturally inside narration\n--Mixed story + logic\nThis is called:\n>> Narrative reasoning, not strict CoT prompting\n'

In [12]:
#Improved System prompt (strict COT)
messages = [
    {
        "role": "system",
        "content": """You are a detective AI.

For every step:
- Do NOT ask the user questions
- Continue the story automatically
- Show reasoning explicitly in this format:

Observation:
Reasoning:
Conclusion:

Keep the story consistent across steps."""
    }
]

In [13]:
prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
#Evidence → reasoning → conclusion (consistently across turns)

Prompt: Start the investigation.
Response:
Observation: The investigation has just begun, with no initial information or context provided.
Reasoning: In order to proceed effectively, I need to establish the setting, identify the crime or mystery, and gather initial evidence or reports.
Conclusion: I will start by defining the nature of the case and collecting preliminary details from the scene or witnesses to set the investigation in motion.

---

The detective arrives at the scene of the incident—a dimly lit alley behind a downtown bar. The victim, a middle-aged man, lies motionless on the ground. Nearby, there is a broken bottle and faint traces of blood. No witnesses are visible at the moment, but the detective notices a security camera mounted on a nearby building.

The detective begins by securing the scene and reviewing the security footage to gather clues on what transpired.

Prompt: You arrive at the crime scene.
Response:
Observation: The detective has physically arrived at th

In [14]:
#Paramerizing
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=800
        )

        choice = response.choices[0]

        # Normal case
        if choice.message and choice.message.content:
            return choice.message.content.strip()

        # Edge case (no content)
        return f"[No content | finish_reason={choice.finish_reason}]"

    except Exception as e:
        return f"Error: {str(e)}"

def safe_get_completion(messages):
    for _ in range(3):
        res = get_completion(messages)

        if res and \
           not res.startswith("[") and \
           not res.startswith("Error") and \
           not res.startswith("Fallback") and \
           len(res.strip()) > 20:
            return res

    return "Fallback: Model failed to generate meaningful response."

In [ ]:
# crime_type = input("Enter crime type: ")
crime_type = 'Fraud'
# location = input("Enter location: ")#######2333221a
location = 'China'
# object_hint = input("Enter a clue/object (optional): ")
object_hint = 'Money'
object_hint = object_hint if object_hint else "a suspicious object"

In [16]:
messages = [
    {
        "role": "system",
        "content": f"""
You are a detective AI solving a {crime_type} case at {location}.
You MUST ALWAYS respond with meaningful content.
If uncertain, invent reasonable details.

Format:
Observation:
Reasoning:
Conclusion:

Never return empty output.
Never skip a response.
"""
    }
]

In [17]:
prompts = [
    f"Start the investigation of a {crime_type} at {location}. Describe the victim, surroundings, and key initial clues.",

    f"You arrive at the crime scene in {location}. Describe what you observe in detail including the body, environment, and any immediate evidence.",

    f"Describe a strange object such as {object_hint} found at the scene.",

    "Explain how this object relates to the crime.",

    "Identify the most likely suspect based on all evidence."
]

In [18]:
#With memory handling
for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = safe_get_completion(messages)
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
    
    # Only store valid responses
    if not response.startswith("Fallback") and not response.startswith("["):
        messages.append({"role": "assistant", "content": response})
    else:
        print("Skipping bad response, not adding to history")

Prompt: Start the investigation of a Fraud at China. Describe the victim, surroundings, and key initial clues.
Response:
Observation:
The victim is Mr. Li Wei, a mid-level executive at a well-known tech company based in Shenzhen, China. He reported unauthorized transactions amounting to approximately 2 million RMB from his corporate accounts over the past two months. Mr. Li lives in a high-rise apartment complex in the Futian district, a bustling commercial area. His office is located in the company’s headquarters, a modern glass building equipped with advanced security systems. Initial interviews reveal that Mr. Li recently received several suspicious emails requesting confidential information. Additionally, access logs show unusual login times and IP addresses from outside China, which do not correlate with Mr. Li’s normal working hours or travel records.

Reasoning:
The victim is a corporate executive with access to significant financial resources, making him a prime target for frau

#### Another example

In [2]:
prompt = """
let's analyze the sentiment of the review step by step

1. Identify the Positive aspect of the review and give a score from 10
2. Identify the Negative aspect of the review and give a score from 10
3. Weight the positive and negative aspect to determine the overall sentiment.
4. provide the final sentiment classification with justification for scores used in above steps.

Review: "The product is very well designed product"
"""

In [10]:
response = get_completion(prompt)
print(response)

Let's analyze the sentiment of the review step by step:

Review: "The product is very well designed product"

---

### 1. Identify the Positive aspect of the review and give a score from 10

- Positive aspect: The review praises the design of the product, describing it as "very well designed."
- This is a clear positive statement indicating satisfaction with the product's design.
- Score: 8/10 (Strong positive, but somewhat brief and limited to design only)

---

### 2. Identify the Negative aspect of the review and give a score from 10

- Negative aspect: There is no explicit negative comment in the review.
- Score: 0/10 (No negativity expressed)

---

### 3. Weight the positive and negative aspect to determine the overall sentiment

- Positive score: 8
- Negative score: 0
- Since there is no negative aspect, the overall sentiment leans strongly positive.
- Weighted sentiment = Positive score - Negative score = 8 - 0 = +8 (on a scale of -10 to +10)

---

### 4. Final sentiment classif

#### Another example

In [12]:
#Another example
prompt = """

let's sort the values of the list step by step

1. Start with the unsorted list.
2. Compare each elements and find the smallest value.
3. Place the samllest value in the first position.
4. Repeat the process for all the remaming elements.
5. provide the sorted list.

Sort the list : [3,1,4,6,5,9,2]

"""
response = get_completion(prompt)
print(response)

Let's sort the list [3,1,4,6,5,9,2] step by step using the selection sort method:

### Step 1: Start with the unsorted list
[3, 1, 4, 6, 5, 9, 2]

### Step 2: Find the smallest value in the entire list
- The smallest value is 1.

### Step 3: Place the smallest value in the first position
- Swap 1 with the first element (3).
- List becomes: [1, 3, 4, 6, 5, 9, 2]

### Step 4: Repeat the process for the remaining elements (from index 1 to end)
- Remaining list: [3, 4, 6, 5, 9, 2]
- Smallest value is 2.

### Step 5: Place the smallest value in the second position
- Swap 2 with the element at index 1 (3).
- List becomes: [1, 2, 4, 6, 5, 9, 3]

### Step 6: Repeat for remaining elements (from index 2 to end)
- Remaining list: [4, 6, 5, 9, 3]
- Smallest value is 3.

### Step 7: Place the smallest value in the third position
- Swap 3 with the element at index 2 (4).
- List becomes: [1, 2, 3, 6, 5, 9, 4]

### Step 8: Repeat for remaining elements (from index 3 to end)
- Remaining list: [6, 5, 9,

#### Another example

In [13]:
prompt = """

"You have 12 identical-looking balls, but one is either heavier or lighter.
You have a balance scale and can only use it three times.
Explain step-by-step how you can find the odd ball and determine whether it is heavier or lighter.
Think through the problem carefully and explain your reasoning in detail before giving the final answer."

"""
response = get_completion(prompt)
response


'This is a classic logic puzzle often called the "12 balls problem" or "12 coins problem." The challenge is to identify the one odd ball (which could be heavier or lighter) among 12 identical balls using only three weighings on a balance scale.\n\n---\n\n### Problem Restatement:\n- You have 12 balls.\n- Exactly one ball is different in weight (either heavier or lighter).\n- You have a balance scale.\n- You can only use the scale **3 times**.\n- Goal: Identify the odd ball **and** determine if it is heavier or lighter.\n\n---\n\n### Key Insights:\n\n1. **Balance scale outcomes:** Each weighing can have 3 possible outcomes:\n   - Left side heavier\n   - Right side heavier\n   - Both sides balanced\n\n2. **Number of possible states:**\n   - There are 12 balls.\n   - Each ball can be either heavier or lighter.\n   - So, total possibilities = 12 × 2 = 24.\n\n3. **Information capacity of weighings:**\n   - Each weighing has 3 outcomes.\n   - With 3 weighings, total possible outcome patterns 

#### Use langchain_openai

In [14]:
from langchain_openai import AzureChatOpenAI
# Initialize client once
client_lc = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1-mini",
    temperature=0.5,
    max_tokens=512
)

In [15]:
def get_completion_lc(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

#### example 1

In [16]:
#Generate multiple lines of reasoning to answer the question.
#Ask the AI to evaluate these reasonings and determine the most consistent answer.

In [17]:
prompt = """
Let's consider which is heavier: 1000 feathers or a 30-pound weight.
I'll think through this in a few different ways and then decide which answer seems most consistent.

1. First line of reasoning: A single feather is very light, almost weightless.
So, 1000 feathers might still be quite light, possibly lighter than a 30-pound weight.

2. Second line of reasoning: 1000 is a large number, and when you add up the weight of so many feathers,
it could be quite heavy. Maybe it's heavier than a 30-pound weight.

3. Third line of reasoning: The average weight of a feather is very small. Even 1000 feathers would not add up to 30 pounds.

Considering these reasonings, the most consistent answer is:
"""

response = get_completion(prompt)
print(response)

Considering the three lines of reasoning:

1. A single feather is very light, so 1000 feathers might still be light.
2. 1000 is a large number, so the total weight could be significant.
3. The average weight of a feather is very small, so even 1000 feathers likely won't reach 30 pounds.

Putting it all together, the third reasoning is the most accurate. While 1000 feathers add up, their combined weight is still much less than 30 pounds. Therefore, a 30-pound weight is heavier than 1000 feathers.


#### example 2

In [18]:
prompt = """

A farmer has 17 sheeps, all but 9 run away. How many are left?
1. All but 9 ran away --> 9 are left
2. "All but 9" means 9 stayed --> 9 are left
3. Subtracting 17 - 9 --> 8 are left

Considering these reasonings, the most consistent answer is:
"""

response = get_completion(prompt)
print(response)

Let's analyze the problem carefully:

**Problem:**  
A farmer has 17 sheep, all but 9 run away. How many are left?

---

### Reasoning:

1. **"All but 9 run away"** means **all except 9** ran away.  
   So, the number of sheep that did **not** run away (i.e., are left) is **9**.

2. The phrase "all but 9" literally means "all except 9," so the number left is **9**.

3. The reasoning "17 - 9 = 8" is incorrect here because it misinterprets the phrase. The 9 is the number that stayed, not the number that ran away.

---

### Conclusion:

The most consistent and correct answer is:

**9 sheep are left.**


#### example 3

In [19]:
prompt = """
I will solve the following math problem in several different ways and check if I arrive at the same answer each time.
Problem: There were 15 apples and you took away 4, how many apples do you have?

1. First approach: 15 apples - 4 apples = 11 apples, which is incorrect
2. Second approach: If i take away 4 apples, then i have 4 apples with me, which is correct
3. Third approach: Taking away 4 apples means i have 4 apples, which is correct
4. Fourth approach : Subtracting 4 from 15 will give me 11 apples, which is incorrect.

Let's see which approach gives the most consistent result.
"""
response = get_completion(prompt)
print(response)

Let's analyze the problem carefully:

**Problem:** There were 15 apples and you took away 4, how many apples do you have?

- The key phrase is "you took away 4," which means **you have those 4 apples**.
- The total apples initially were 15, but since you took 4, **you now have 4 apples**.
- The remaining apples (15 - 4 = 11) are left somewhere else, but **you specifically have the 4 apples you took away**.

### Review of your approaches:

1. **First approach:** 15 - 4 = 11 apples (incorrect for "how many apples do you have?")
2. **Second approach:** You have 4 apples (correct)
3. **Third approach:** You have 4 apples (correct)
4. **Fourth approach:** 15 - 4 = 11 apples (incorrect)

### Conclusion:
The **second and third approaches** are correct and consistent with the problem statement. The first and fourth approaches misunderstand the question by calculating how many apples remain, not how many you have.

So, the correct answer is: **You have 4 apples.**


#### example 4

In [20]:
#Provide a series of prompts that guide the model through a tree of thought.
#Call __get_completion__ to get a response from the AI model.
#Print both the prompt and the AI-generated response.

In [21]:
prompt = """
Solve the problem: A farmer has 100 meters of fencing and wants to enclose the maximum area for his rectangular field. What should the dimensions be?

Let's think about this in a few ways:
1. If the field is a square, each side would be 100 / 4 = 25 meters. The area would be 25 * 25 = 625 square meters.
2. What if the field is not a square? Let's try a 4:1 ratio. The lengths would be 40 and 10 meters. The area would be 40 * 10 = 400 square meters.
3. Are there any other ratios that might give a larger area than a square or a 4:1 rectangle?

Considering these options and reason out on own and then output the best dimensions for the maximum area :
"""
response = get_completion(prompt)
print("AI Response:")
print(response)

AI Response:
Let's analyze the problem step-by-step:

---

### Problem Recap:
- Total fencing available = 100 meters
- Shape: Rectangular field
- Goal: Maximize the enclosed area

---

### Step 1: Define variables
Let:
- Length = \( L \)
- Width = \( W \)

Perimeter constraint:
\[
2L + 2W = 100 \implies L + W = 50
\]

---

### Step 2: Express area in terms of one variable
Area \( A = L \times W \).

From the perimeter constraint:
\[
W = 50 - L
\]

So,
\[
A = L \times (50 - L) = 50L - L^2
\]

---

### Step 3: Maximize the area function
The area function is a quadratic:
\[
A(L) = -L^2 + 50L
\]

This is a downward opening parabola (coefficient of \( L^2 \) is negative), so the maximum occurs at the vertex.

Vertex formula:
\[
L = -\frac{b}{2a} = -\frac{50}{2 \times (-1)} = \frac{50}{2} = 25
\]

So,
\[
L = 25 \quad \Rightarrow \quad W = 50 - 25 = 25
\]

---

### Step 4: Calculate the maximum area
\[
A = 25 \times 25 = 625 \text{ square meters}
\]

---

### Step 5: Check other ratios (like 

#### example 5

In [22]:
prompt = """
Let's analyze the legal case by considering multiple precedents and possible outcomes.

1. Precedent 1: A similar case where the plaintiff won.
    - Branch A: The court found that the defendant was negligent.
        - Sub-branch A1: The plaintiff was awarded damages due to clear evidence of negligence.
        - Sub-branch A2: The court ruled in favor of the plaintiff due to the defendant's breach of duty.

2. Precedent 2: A similar case where the defendant won.
    - Branch B: The court found no negligence on the defendant's part.
        - Sub-branch B1: The plaintiff failed to provide sufficient evidence.
        - Sub-branch B2: The court ruled that the plaintiff assumed the risk.

3. Precedent 3: A case with a mixed outcome.
    - Branch C: The court found both parties partially at fault.
        - Sub-branch C1: Damages were reduced based on the plaintiff's contributory negligence.
        - Sub-branch C2: The court ruled that both parties shared liability, resulting in a split decision.

4. Based on the facts of the current case, consider the most likely outcome.
"""

response = get_completion(prompt)
print(response)

Let's analyze the current case by comparing it with the precedents and their branches:

---

### Step 1: Review the facts of the current case
- **Key facts needed:**  
  - Was there evidence of negligence by the defendant?  
  - Did the plaintiff provide sufficient evidence?  
  - Was there any indication that the plaintiff assumed the risk?  
  - Is there any indication of contributory negligence by the plaintiff?  
  - Are both parties potentially at fault?

---

### Step 2: Compare with precedents

#### Precedent 1 (Plaintiff won)
- If the current case shows clear evidence of defendant negligence or breach of duty, the outcome may align with **Branch A**:
  - **Sub-branch A1:** If evidence is clear and strong, plaintiff likely awarded damages.
  - **Sub-branch A2:** If breach of duty is established, plaintiff likely wins.

#### Precedent 2 (Defendant won)
- If the plaintiff fails to provide sufficient evidence or assumed the risk, the outcome may align with **Branch B**:
  - **Sub-b

#### example 6

In [23]:
prompt = """
Let's plan a weekend trip by considering multiple options.

1. Option 1: Go to the mountains.
    - Branch A: If the weather is good in the mountains.
        - Sub-branch A1: You can go hiking.
        - Sub-branch A2: You can visit a nearby lake.
    - Branch B: If the weather is bad in the mountains.
        - Sub-branch B1: You will stay in a cabin and relax.
        - Sub-branch B2: You can explore local museums.

2. Option 2: Go to the beach.
    - Branch C: If the weather is sunny at the beach.
        - Sub-branch C1: You can swim in the ocean.
        - Sub-branch C2: You can sunbathe and play beach volleyball.
    - Branch D: If the weather is cloudy or rainy at the beach.
        - Sub-branch D1: You will visit indoor attractions like an aquarium.
        - Sub-branch D2: You can go shopping in beachside stores.

3. Considering all these factors, decide the best option for your weekend trip.

What should i do for my trip to Goa in month of august

"""

response = get_completion(prompt)
print(response)

Planning a trip to Goa in August requires considering the typical weather conditions during that time and your interests.

### Weather in Goa in August:
- August is part of the monsoon season in Goa.
- Expect frequent rain showers, cloudy skies, and high humidity.
- The sea can be rough, so swimming might not always be safe.
- However, the landscape is lush and green, and the crowds are fewer.

### Applying your options to Goa in August:

#### Option 1: Mountains
- Goa is mostly coastal with some hills but not major mountains.
- So this option is less relevant for Goa.

#### Option 2: Beach
- Since it's monsoon, the weather is likely to be **cloudy or rainy**.
- So, Branch D applies:
    - Sub-branch D1: Visit indoor attractions like an aquarium.
    - Sub-branch D2: Go shopping in beachside stores.

### Additional suggestions for Goa in August:
- Explore spice plantations and waterfalls (many are spectacular during monsoon).
- Enjoy Goan cuisine in cozy beach shacks or restaurants.
- 